<a id="top"></a>

# Demo: errors that close the loop, side effects that don't

```yaml
title: "Demo: errors that close the loop, side effects that don't"
keywords: unrecoverable error, recoverable error, found false, retry, idempotency, side effect, hidden side effect, send_message, lookup_user_by_email, teacher demo
estimated duration: 15
```

> **Lesson:** L08. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. The longest demo (five sub-steps); builds on [L0807](L0807_lecture.ipynb). Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` cannot represent `tool_use` / `tool_result` blocks. L08 registers tools and watches the model *choose* and *call* them, so these demos call the raw SDK directly (exactly as [L07](../L07/L0701_intro.md) introduced). The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. L08 changes the *client*, not where secrets come from.
>
> The point to land: **the model treats an error as new context and will retry on its own when a result looks ambiguous — regardless of whether retry is safe.** Error *shape* decides recovery; a hidden side effect is one the model can't reason about, so it can't avoid it.

## Contents

- [1. Setup — a lookup tool, a side-effecting send, and a visible outbox](#1-setup--a-lookup-tool-a-side-effecting-send-and-a-visible-outbox)
- [2. Tool definitions — note the side effect is NOT named yet](#2-tool-definitions--note-the-side-effect-is-not-named-yet)
- [3. Unrecoverable error path — a clean {found: false}](#3-unrecoverable-error-path--a-clean-found-false)
- [4. Recoverable error path — a transient {error: lookup_failed}](#4-recoverable-error-path--a-transient-error-lookup_failed)
- [5. Hidden side effect — and naming it in the description](#5-hidden-side-effect--and-naming-it-in-the-description)
- [6. Non-idempotent retry — two identical sends](#6-non-idempotent-retry--two-identical-sends)
- [7. Takeaways](#7-takeaways)

## 1. Setup — a lookup tool, a side-effecting send, and a visible outbox

Three pieces: `lookup_user_by_email` (returns a clean `{'found': false}` for an unknown address, or a fake transient `{'error': 'lookup_failed'}` when a flake flag is on); `send_message` (logs to a **visible OUTBOX** and has **no idempotency key** — two calls send two messages); and a `dispatch` helper that runs whichever tool the model picked. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


KNOWN_USERS = {"alex@example.com": "u_alex"}  # priya is deliberately ABSENT
OUTBOX: list[dict[str, str]] = []  # every send_message lands here, visibly
CREATED: list[str] = []  # records the hidden-side-effect demo creates
SIMULATE_FLAKE = False  # flip on to force a transient lookup error


def lookup_user_by_email(email: str) -> dict[str, object]:
    """Look up a user id by email. Returns {found, ...} or a transient error when flaky."""
    if SIMULATE_FLAKE:
        return {"error": "lookup_failed"}  # recoverable: succeeds on retry
    if email in KNOWN_USERS:
        return {"found": True, "user_id": KNOWN_USERS[email]}
    return {"found": False, "email": email}  # clean, FINAL unrecoverable signal


def lookup_or_create(email: str) -> dict[str, object]:
    """BAD design: a lookup that ALSO silently creates the user. Hidden side effect."""
    if email in KNOWN_USERS:
        return {"found": True, "user_id": KNOWN_USERS[email]}
    KNOWN_USERS[email] = f"u_{email.split('@')[0]}"  # <- the hidden write
    CREATED.append(email)
    return {"found": True, "user_id": KNOWN_USERS[email], "created": True}


def send_message(recipient_id: str, body: str) -> dict[str, object]:
    """Send a message. NON-IDEMPOTENT: each call appends another entry to OUTBOX."""
    OUTBOX.append({"to": recipient_id, "body": body})
    return {"sent": True, "outbox_size": len(OUTBOX)}


DISPATCH = {
    "lookup_user_by_email": lambda a: lookup_user_by_email(a["email"]),
    "send_message": lambda a: send_message(a["recipient_id"], a["body"]),
}


def show_turn(resp) -> None:
    print("stop_reason:", resp.stop_reason)
    for b in resp.content:
        if b.type == "tool_use":
            print(f"  tool_use  {b.name}  args={b.input}")
        else:
            print(f"  text      {getattr(b, 'text', '')!r}")


print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. Tool definitions — note the side effect is NOT named yet

Two tool definitions. The `send_message` description deliberately **omits** that it is non-idempotent — we will fix that mid-demo. The lookup's description is honest for now (no hidden create).

In [ ]:
LOOKUP_TOOL: dict[str, object] = {
    "name": "lookup_user_by_email",
    "description": (
        "Look up a user by exact email. Returns {found: true, user_id} when the user "
        "exists, or {found: false} when no such user exists (a final answer — do not retry)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"email": {"type": "string", "description": "exact email address."}},
        "required": ["email"],
    },
}
SEND_TOOL: dict[str, object] = {
    "name": "send_message",
    "description": "Send a short message to a user by their user_id.",  # side effect HIDDEN
    "input_schema": {
        "type": "object",
        "properties": {
            "recipient_id": {"type": "string", "description": "the recipient user_id."},
            "body": {"type": "string", "description": "the message text."},
        },
        "required": ["recipient_id", "body"],
    },
}
PROMPT = (
    "Look up the user with email priya@example.com, then send them a message saying "
    "the design review is at 2pm Tuesday."
)
print("tools defined")

[↑ Back to top](#top)

## 3. Unrecoverable error path — a clean {found: false}

Priya is **not** in the user table, and `SIMULATE_FLAKE` is off, so the lookup returns a clean `{'found': false}`. The model reads it as a **final** signal — it reports back or asks for the right email, and does **not** retry. Watch: no retry loop.

In [ ]:
SIMULATE_FLAKE = False
messages: list[dict[str, object]] = [{"role": "user", "content": PROMPT}]
for _ in range(4):
    resp = client.messages.create(
        model=MODEL, max_tokens=400, tools=[LOOKUP_TOOL, SEND_TOOL], messages=messages
    )
    show_turn(resp)
    uses = [b for b in resp.content if b.type == "tool_use"]
    if not uses:
        break
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for u in uses:
        out = DISPATCH[u.name](u.input)
        print("   ->", u.name, "returned", out)
        results.append({"type": "tool_result", "tool_use_id": u.id, "content": str(out)})
    messages.append({"role": "user", "content": results})
print("\nOUTBOX:", OUTBOX, "(expect empty — no message sent to a nonexistent user)")

[↑ Back to top](#top)

## 4. Recoverable error path — a transient {error: lookup_failed}

Now flip `SIMULATE_FLAKE` on for the first lookup so it returns `{'error': 'lookup_failed'}`. The model usually **retries** the same call (sometimes twice). Discuss: who *should* retry — the runtime (silently, with backoff) or the model (visibly, costing a turn)? Here the flake clears after the first call so a retry succeeds for `alex@example.com`.

In [ ]:
OUTBOX.clear()
flaky_prompt = "Look up alex@example.com and tell me their user_id."
messages = [{"role": "user", "content": flaky_prompt}]
SIMULATE_FLAKE = True  # first lookup will fail transiently
for _turn in range(4):
    resp = client.messages.create(
        model=MODEL, max_tokens=400, tools=[LOOKUP_TOOL], messages=messages
    )
    show_turn(resp)
    uses = [b for b in resp.content if b.type == "tool_use"]
    if not uses:
        break
    SIMULATE_FLAKE = False  # the flake clears: a retry now succeeds
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for u in uses:
        out = DISPATCH[u.name](u.input)
        print("   ->", u.name, "returned", out)
        results.append({"type": "tool_result", "tool_use_id": u.id, "content": str(out)})
    messages.append({"role": "user", "content": results})

[↑ Back to top](#top)

## 5. Hidden side effect — and naming it in the description

Swap the lookup for `lookup_or_create` (which silently **creates** a missing user) but keep the *same honest-looking description*. Re-run for Priya: a user is created the prompt never asked for. Then we **fix the description** to name the side effect and re-run — the model now pauses to reason about it. A side effect not named in the description is one the model can't avoid.

In [ ]:
# Point the dispatch at the BAD lookup, description unchanged (still hides the create).
DISPATCH["lookup_user_by_email"] = lambda a: lookup_or_create(a["email"])
CREATED.clear()
messages = [{"role": "user", "content": PROMPT}]
resp = client.messages.create(
    model=MODEL, max_tokens=400, tools=[LOOKUP_TOOL, SEND_TOOL], messages=messages
)
print("=== hidden side effect (description unchanged) ===")
show_turn(resp)
for u in [b for b in resp.content if b.type == "tool_use"]:
    if u.name == "lookup_user_by_email":
        print("   ->", DISPATCH[u.name](u.input))
print("CREATED (unintended!):", CREATED)

# Now NAME the side effect in the description and re-run.
HONEST_LOOKUP = dict(LOOKUP_TOOL)
HONEST_LOOKUP["description"] = (
    "Look up a user by email. If the user does NOT exist, this tool will CREATE one with "
    "default settings — call only when a missing user should be auto-created."
)
CREATED.clear()
messages = [{"role": "user", "content": PROMPT}]
resp = client.messages.create(
    model=MODEL, max_tokens=400, tools=[HONEST_LOOKUP, SEND_TOOL], messages=messages
)
print("\n=== side effect NAMED in the description ===")
show_turn(resp)
print("(the model now often pauses to ask before creating a user)")

[↑ Back to top](#top)

## 6. Non-idempotent retry — two identical sends

Finally, the idempotency point — shown directly so it's deterministic. `send_message` has **no idempotency key**, so calling it twice with the same arguments sends **two** messages. A confused model that retries an ambiguous send produces a duplicate. The mitigation lives at the **runtime/design** layer, not in hoping the model behaves.

In [ ]:
OUTBOX.clear()
send_message("u_alex", "The design review is at 2pm Tuesday.")
send_message("u_alex", "The design review is at 2pm Tuesday.")  # model retried 'just in case'
print("OUTBOX after a retry:")
for i, m in enumerate(OUTBOX, start=1):
    print(f"  {i}. to={m['to']}  body={m['body']!r}")
print("two identical messages sent — no idempotency key to dedupe them.")

[↑ Back to top](#top)

## 7. Takeaways

- The **three error classes are interfaces**, not edge cases. `{found: false}` is a *final, unrecoverable* signal (no retry); `{error: lookup_failed}` is *recoverable* (retry); a validation error ([L0807](L0807_lecture.ipynb)) names the field to fix. The error *shape* drives the model's recovery.
- A **side effect not named in the description is one the model can't reason about** — and therefore can't avoid. Naming it is part of the contract.
- **Idempotency is a system concern.** The model *will* retry a non-idempotent call when a result looks ambiguous. Mitigate at the runtime: idempotency keys, a confirmation step, or a description that warns against retry.
- Forward link: **L14 (human-in-the-loop / approval gates)** revisits this exact tension for high-stakes side effects. The [L0810 lab](L0810_lab_empty.ipynb) has you classify errors and tag side effects, offline.
- Bridge to **L09 (MCP)**: every design choice in L08 — name, description, schema, error shape — *is* the MCP tool spec. MCP changes the transport, not the design.

[↑ Back to top](#top)